# RAGAS Advanced -- Testset Generation, Custom Metrics & More

This notebook covers advanced RAGAS features that go beyond basic metric evaluation:

1. **Testset Generation** -- Automatically generate evaluation data from documents
2. **Noise Sensitivity** -- Measure how irrelevant context affects answers
3. **Semantic Similarity** -- Embedding-based answer comparison
4. **Custom Metrics** -- Build your own evaluation metrics
5. **LangChain Integration** -- Evaluate LangChain chains with RAGAS
6. **Comparing RAG Configurations** -- Test different chunk sizes, top-K values
7. **RAGAS vs DeepEval** -- Side-by-side comparison

**Prerequisites**: Notebook 06 completed, OpenAI API key set in `.env`

In [ ]:
import os
import json
import pandas as pd
import numpy as np
from dotenv import load_dotenv

load_dotenv(os.path.join("..", ".env"))
assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in your .env file"

from openai import OpenAI
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

print("Base imports loaded.")

In [ ]:
# RAGAS imports
from ragas import evaluate, EvaluationDataset, SingleTurnSample
from ragas.metrics import (
    Faithfulness,
    ResponseRelevancy,
    LLMContextPrecisionWithReference,
    LLMContextRecall,
    FactualCorrectness,
    SemanticSimilarity,
    NoiseSensitivity,
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# Evaluator LLM and embeddings
evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini", temperature=0))
evaluator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))

print("RAGAS imports and evaluator configured.")

## Setup: Rebuild the RAG Pipeline

We will use the same Acme Corp knowledge base and RAG pipeline from notebook 06.

In [ ]:
# Initialize clients
openai_client = OpenAI()
qdrant_client = QdrantClient(":memory:")

EMBEDDING_MODEL = "text-embedding-3-small"
GENERATION_MODEL = "gpt-4o-mini"
EMBEDDING_DIM = 1536

# Acme Corp knowledge base
ACME_DOCUMENTS = [
    {
        "id": 1,
        "title": "Return Policy Overview",
        "content": (
            "Acme Corp offers a 30-day return policy on all products purchased through "
            "our website or retail stores. To be eligible for a return, the item must be "
            "unused, in its original packaging, and accompanied by the original receipt or "
            "proof of purchase. Refunds are processed within 5-7 business days after we "
            "receive the returned item. Shipping costs for returns are the responsibility "
            "of the customer unless the item was defective or the wrong item was shipped. "
            "Items marked as 'Final Sale' cannot be returned or exchanged."
        ),
    },
    {
        "id": 2,
        "title": "Electronics Return Policy",
        "content": (
            "Electronic products purchased from Acme Corp have a 15-day return window "
            "instead of the standard 30 days. All electronics must be returned with all "
            "original accessories, cables, manuals, and packaging. A restocking fee of 15% "
            "may apply to opened electronics. Defective electronics can be exchanged for "
            "the same model within the first 90 days at no additional cost."
        ),
    },
    {
        "id": 3,
        "title": "Shipping Policy",
        "content": (
            "Acme Corp offers several shipping options for domestic orders. Standard Shipping "
            "takes 5-7 business days and is free for orders over $50. Expedited Shipping takes "
            "2-3 business days and costs $12.99. Overnight Shipping is available for $24.99 and "
            "delivers the next business day if ordered before 2 PM EST."
        ),
    },
    {
        "id": 4,
        "title": "International Shipping Policy",
        "content": (
            "Acme Corp ships internationally to over 50 countries. International Standard "
            "Shipping takes 10-21 business days. International Express takes 5-7 business days. "
            "All international orders may be subject to customs duties and import fees which "
            "are the customer's responsibility."
        ),
    },
    {
        "id": 5,
        "title": "Product Warranty Information",
        "content": (
            "All Acme Corp branded products come with a 1-year limited warranty covering "
            "defects in materials and workmanship under normal use. The warranty does not "
            "cover accidents, misuse, or normal wear and tear. Extended warranty plans "
            "(2-year and 3-year) are available for an additional fee."
        ),
    },
    {
        "id": 6,
        "title": "Customer Support Channels",
        "content": (
            "Acme Corp customer support is available by phone (1-800-ACME-HELP, M-F 8AM-8PM "
            "EST), email (support@acmecorp.com, 24-48hr response), and live chat (M-Sat "
            "9AM-6PM EST). An automated help center is available at help.acmecorp.com."
        ),
    },
    {
        "id": 7,
        "title": "Accepted Payment Methods",
        "content": (
            "Acme Corp accepts Visa, MasterCard, American Express, Discover, PayPal, Apple Pay, "
            "Google Pay, and Acme Gift Cards. For orders over $200, Acme Pay Later splits "
            "payments into 4 interest-free installments over 6 weeks. Annual contracts for "
            "business accounts receive a 20% discount."
        ),
    },
    {
        "id": 8,
        "title": "Acme Rewards Loyalty Program",
        "content": (
            "Acme Rewards is free to join, earning 1 point per dollar spent. 100 points = $5 off. "
            "Benefits include early sale access, birthday 20% discount, and free standard shipping. "
            "Silver tier (500+ pts/year) adds free expedited shipping. Gold tier (1000+ pts/year) "
            "adds priority customer support. Points expire after 12 months of inactivity."
        ),
    },
]

print(f"Loaded {len(ACME_DOCUMENTS)} documents")

In [ ]:
# Helper functions
def get_embeddings(texts, model=EMBEDDING_MODEL):
    """Get embeddings for a list of texts."""
    response = openai_client.embeddings.create(input=texts, model=model)
    return [item.embedding for item in response.data]

def build_qdrant_collection(documents, collection_name, chunk_size=None):
    """Build a Qdrant collection from documents.
    
    If chunk_size is provided, split documents into chunks of that character length.
    Otherwise, use each document as a single chunk.
    """
    chunks = []
    chunk_id = 1
    
    for doc in documents:
        text = f"{doc['title']}\n\n{doc['content']}"
        if chunk_size and len(text) > chunk_size:
            # Simple character-based chunking with overlap
            overlap = min(50, chunk_size // 4)
            start = 0
            while start < len(text):
                end = start + chunk_size
                chunk_text = text[start:end]
                chunks.append({"id": chunk_id, "text": chunk_text, "title": doc["title"]})
                chunk_id += 1
                start = end - overlap
        else:
            chunks.append({"id": chunk_id, "text": text, "title": doc["title"]})
            chunk_id += 1
    
    # Embed
    chunk_texts = [c["text"] for c in chunks]
    embeddings = get_embeddings(chunk_texts)
    
    # Create collection
    if collection_name in [c.name for c in qdrant_client.get_collections().collections]:
        qdrant_client.delete_collection(collection_name)
    
    qdrant_client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=EMBEDDING_DIM, distance=Distance.COSINE),
    )
    
    points = [
        PointStruct(id=c["id"], vector=emb, payload={"text": c["text"], "title": c["title"]})
        for c, emb in zip(chunks, embeddings)
    ]
    qdrant_client.upsert(collection_name=collection_name, points=points)
    return len(chunks)

def retrieve(query, collection_name, top_k=3):
    """Retrieve top-k relevant chunks."""
    query_embedding = get_embeddings([query])[0]
    results = qdrant_client.search(
        collection_name=collection_name,
        query_vector=query_embedding,
        limit=top_k,
    )
    return [hit.payload["text"] for hit in results]

def generate(query, contexts, temperature=0.0):
    """Generate answer from query and contexts."""
    context_str = "\n\n---\n\n".join(contexts)
    messages = [
        {"role": "system", "content": (
            "You are a helpful customer support assistant for Acme Corp. "
            "Answer based on the provided context. "
            "If the answer is not in the context, say so. Be concise."
        )},
        {"role": "user", "content": f"Context:\n{context_str}\n\nQuestion: {query}"},
    ]
    response = openai_client.chat.completions.create(
        model=GENERATION_MODEL, messages=messages, temperature=temperature,
    )
    return response.choices[0].message.content

def rag_pipeline(query, collection_name="acme_default", top_k=3, temperature=0.0):
    """Full RAG pipeline."""
    contexts = retrieve(query, collection_name, top_k=top_k)
    answer = generate(query, contexts, temperature=temperature)
    return {"query": query, "answer": answer, "contexts": contexts}

# Build default collection
n_chunks = build_qdrant_collection(ACME_DOCUMENTS, "acme_default")
print(f"Built default collection with {n_chunks} chunks")

-
## Part 1: Testset Generation

RAGAS can **automatically generate evaluation test sets** from your documents. This is incredibly valuable because:
- Manually creating golden test sets is expensive and time-consuming
- Generated test sets cover different difficulty levels and question types
- You can generate hundreds of test cases automatically

RAGAS TestsetGenerator creates questions with different **evolution types**:
- **Simple**: Direct factual questions that can be answered from a single chunk
- **Reasoning**: Questions requiring inference or multi-step reasoning
- **Multi-context**: Questions that require combining information from multiple chunks

In [ ]:
from ragas.testset import TestsetGenerator
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.documents import Document as LCDocument

# Convert our documents to LangChain Document format for the generator
lc_documents = [
    LCDocument(
        page_content=f"{doc['title']}\n\n{doc['content']}",
        metadata={"title": doc["title"], "id": doc["id"]},
    )
    for doc in ACME_DOCUMENTS
]

print(f"Prepared {len(lc_documents)} LangChain documents for testset generation")

In [ ]:
# Initialize the TestsetGenerator
generator_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)
generator_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings,
)

print("TestsetGenerator initialized.")

In [ ]:
# Generate a test set from our documents
# We request 10 test samples across different query types
testset = generator.generate_with_langchain_docs(
    documents=lc_documents,
    testset_size=10,
)

print(f"Generated {len(testset)} test samples")

In [ ]:
# Inspect the generated test cases
testset_df = testset.to_pandas()
print("Generated Test Cases:")
print("=" * 80)
for idx, row in testset_df.iterrows():
    print(f"\n--- Test Case {idx + 1} ---")
    print(f"  Question: {row.get('user_input', 'N/A')}")
    print(f"  Reference: {str(row.get('reference', 'N/A'))[:120]}...")
    print(f"  Contexts: {len(row.get('reference_contexts', []))} context(s)")

testset_df.head()

In [ ]:
# Now evaluate our RAG pipeline on the generated test set
# First, run each generated question through our RAG pipeline
generated_samples = []

for idx, row in testset_df.iterrows():
    question = row["user_input"]
    result = rag_pipeline(question)
    
    sample = SingleTurnSample(
        user_input=question,
        response=result["answer"],
        reference=str(row.get("reference", "")),
        retrieved_contexts=result["contexts"],
    )
    generated_samples.append(sample)
    print(f"Processed question {idx + 1}: {question[:60]}...")

generated_eval_dataset = EvaluationDataset(samples=generated_samples)
print(f"\nCreated evaluation dataset with {len(generated_eval_dataset)} samples")

In [ ]:
# Evaluate with core metrics on the generated test set
generated_results = evaluate(
    dataset=generated_eval_dataset,
    metrics=[
        Faithfulness(llm=evaluator_llm),
        ResponseRelevancy(llm=evaluator_llm, embeddings=evaluator_embeddings),
        LLMContextRecall(llm=evaluator_llm),
    ],
)

gen_results_df = generated_results.to_pandas()
print("Results on Generated Test Set:")
print("=" * 60)
print(f"  Faithfulness (mean):      {gen_results_df['faithfulness'].mean():.3f}")
print(f"  Answer Relevancy (mean):  {gen_results_df['answer_relevancy'].mean():.3f}")
print(f"  Context Recall (mean):    {gen_results_df['llm_context_recall'].mean():.3f}")
gen_results_df

-
## Part 2: Noise Sensitivity

**Noise Sensitivity** measures how much the RAG system's output is affected by **irrelevant or noisy context**. In real-world retrieval, not all retrieved chunks are relevant -- some are noise.

A robust RAG system should:
- Ignore irrelevant retrieved chunks
- Not incorporate noise into its answers
- Maintain answer quality even with imperfect retrieval

The metric works by comparing the system's behavior with and without noisy context, measuring whether irrelevant information degrades the answer.

In [ ]:
# Create test cases with different noise levels
# We will manually inject irrelevant context alongside relevant context

irrelevant_contexts = [
    "The weather in Paris is typically mild in spring with average temperatures of 15C. "
    "The Eiffel Tower was built in 1889 for the World Fair.",
    "Python was created by Guido van Rossum and first released in 1991. It emphasizes "
    "code readability with its use of significant whitespace.",
    "The Great Wall of China stretches over 13,000 miles and was built over many centuries "
    "to protect Chinese states from invasions.",
]

noise_test_cases = [
    {
        "query": "What is Acme Corp's return policy?",
        "reference": "Acme Corp offers a 30-day return policy on all products. Items must be unused, in original packaging, with receipt.",
    },
    {
        "query": "How much does overnight shipping cost?",
        "reference": "Overnight Shipping is available for $24.99 and delivers the next business day if ordered before 2 PM EST.",
    },
    {
        "query": "What warranty do Acme products come with?",
        "reference": "All Acme Corp branded products come with a 1-year limited warranty covering defects in materials and workmanship.",
    },
]

# Build samples with noise injected into retrieved contexts
noise_samples = []
for tc in noise_test_cases:
    # Get real retrieval results
    real_contexts = retrieve(tc["query"], "acme_default", top_k=2)
    # Add irrelevant noise context
    noisy_contexts = real_contexts + irrelevant_contexts
    # Generate answer with noisy context
    answer = generate(tc["query"], noisy_contexts)
    
    sample = SingleTurnSample(
        user_input=tc["query"],
        response=answer,
        reference=tc["reference"],
        retrieved_contexts=noisy_contexts,
    )
    noise_samples.append(sample)
    print(f"Created noisy sample: {tc['query']}")
    print(f"  Contexts: {len(real_contexts)} relevant + {len(irrelevant_contexts)} noise = {len(noisy_contexts)} total")

noise_dataset = EvaluationDataset(samples=noise_samples)

In [ ]:
# Evaluate Noise Sensitivity
noise_results = evaluate(
    dataset=noise_dataset,
    metrics=[
        NoiseSensitivity(llm=evaluator_llm),
        Faithfulness(llm=evaluator_llm),
    ],
)

noise_df = noise_results.to_pandas()
print("Noise Sensitivity Results:")
print("=" * 60)
for idx, row in noise_df.iterrows():
    print(f"\n  Query: {row['user_input']}")
    print(f"  Noise Sensitivity: {row.get('noise_sensitivity', 'N/A')}")
    print(f"  Faithfulness:      {row['faithfulness']:.3f}")

noise_df

### Interpreting Noise Sensitivity

- **Low noise sensitivity** (closer to 0): The system is robust -- it ignores irrelevant context
- **High noise sensitivity** (closer to 1): The system is fragile -- irrelevant context degrades answers

If your system shows high noise sensitivity, consider:
1. Adding a reranker to filter out irrelevant chunks
2. Reducing top-K to retrieve fewer (but more relevant) chunks
3. Improving the system prompt to focus on relevant information

-
## Part 3: Semantic Similarity Metric

Semantic Similarity measures the embedding-based similarity between the generated answer and the reference answer. Unlike factual correctness (which checks individual claims), semantic similarity looks at the overall meaning.

In [ ]:
# Create test cases with varying degrees of semantic similarity
semantic_test_cases = [
    # High similarity -- paraphrase of reference
    SingleTurnSample(
        user_input="What is Acme Corp's return policy?",
        response="Acme Corp provides a 30-day return window for all items bought through their website or stores. Items must be unused and in original packaging.",
        reference="Acme Corp offers a 30-day return policy on all products purchased through our website or retail stores. Items must be unused, in original packaging, with receipt.",
        retrieved_contexts=["Acme Corp offers a 30-day return policy on all products purchased through our website or retail stores."],
    ),
    # Medium similarity -- correct but different phrasing
    SingleTurnSample(
        user_input="What does the warranty cover?",
        response="Acme's warranty protects against manufacturing issues for one year after purchase.",
        reference="All Acme Corp branded products come with a 1-year limited warranty covering defects in materials and workmanship under normal use.",
        retrieved_contexts=["All Acme Corp branded products come with a 1-year limited warranty covering defects in materials and workmanship."],
    ),
    # Low similarity -- partially wrong
    SingleTurnSample(
        user_input="How much does shipping cost?",
        response="Shipping at Acme Corp is always free on all orders with no minimum purchase required.",
        reference="Standard Shipping is free for orders over $50. Expedited Shipping costs $12.99. Overnight Shipping costs $24.99.",
        retrieved_contexts=["Standard Shipping takes 5-7 business days and is free for orders over $50."],
    ),
]

semantic_dataset = EvaluationDataset(samples=semantic_test_cases)

# Run Semantic Similarity
semantic_results = evaluate(
    dataset=semantic_dataset,
    metrics=[SemanticSimilarity(embeddings=evaluator_embeddings)],
)

sem_df = semantic_results.to_pandas()
print("Semantic Similarity Results:")
print("=" * 60)
for idx, row in sem_df.iterrows():
    print(f"\n  Query: {row['user_input']}")
    print(f"  Score: {row['semantic_similarity']:.3f}")

sem_df[["user_input", "semantic_similarity"]]

-
## Part 4: Custom Metrics in RAGAS

RAGAS allows you to create custom evaluation metrics. This is useful when:
- Standard metrics do not capture your domain-specific quality requirements
- You need to evaluate a custom dimension (e.g., tone, formatting, citation quality)
- You want an LLM-as-judge metric with custom criteria

### Option A: Custom Metric with SingleTurnMetric

This is a programmatic metric that computes a score using custom logic.

In [ ]:
from ragas.metrics.base import SingleTurnMetric, MetricType
from dataclasses import dataclass, field
import typing as t


@dataclass
class ResponseConcisenessMetric(SingleTurnMetric):
    """Custom metric that measures how concise the response is.
    
    Penalizes overly verbose responses. The ideal response length
    is between min_words and max_words. Scores decrease for responses
    that are too short or too long.
    """
    name: str = "response_conciseness"
    _required_columns: t.Dict[MetricType, t.Set[str]] = field(
        default_factory=lambda: {MetricType.SINGLE_TURN: {"response"}}
    )
    min_words: int = 10
    max_words: int = 100
    
    async def _single_turn_ascore(self, sample: SingleTurnSample, callbacks=None) -> float:
        """Compute conciseness score for a single sample."""
        response = sample.response
        word_count = len(response.split())
        
        if word_count < self.min_words:
            # Too short -- penalize proportionally
            return max(0.0, word_count / self.min_words)
        elif word_count <= self.max_words:
            # In the sweet spot
            return 1.0
        else:
            # Too long -- penalize proportionally
            excess_ratio = (word_count - self.max_words) / self.max_words
            return max(0.0, 1.0 - excess_ratio)


# Test the custom metric
conciseness_metric = ResponseConcisenessMetric()

test_samples = [
    SingleTurnSample(
        user_input="What is Acme Corp's return policy?",
        response="Acme Corp offers a 30-day return policy on all products purchased through their website or retail stores.",
        retrieved_contexts=["Acme Corp offers a 30-day return policy."],
    ),
    SingleTurnSample(
        user_input="What is Acme Corp's return policy?",
        response="30-day returns allowed.",
        retrieved_contexts=["Acme Corp offers a 30-day return policy."],
    ),
    SingleTurnSample(
        user_input="What is Acme Corp's return policy?",
        response=(
            "Acme Corp offers a comprehensive 30-day return policy that applies to all products "
            "purchased through our website or any of our retail store locations across the country. "
            "To be eligible for a return, the item must be completely unused, in its original "
            "packaging with all tags attached, and accompanied by the original receipt or proof of "
            "purchase. Refunds are processed within 5-7 business days after we receive the returned "
            "item at our processing center. The shipping costs for returns are the responsibility "
            "of the customer unless the item was defective upon arrival or the wrong item was "
            "shipped to the customer. Please note that items marked as Final Sale cannot be returned "
            "or exchanged under any circumstances."
        ),
        retrieved_contexts=["Acme Corp offers a 30-day return policy."],
    ),
]

conciseness_dataset = EvaluationDataset(samples=test_samples)

conciseness_results = evaluate(
    dataset=conciseness_dataset,
    metrics=[conciseness_metric],
)

conc_df = conciseness_results.to_pandas()
print("Response Conciseness Scores:")
print("=" * 60)
labels = ["Concise answer", "Too short", "Too verbose"]
for idx, (row, label) in enumerate(zip(conc_df.itertuples(), labels)):
    print(f"\n  [{label}]")
    print(f"  Response: {row.response[:80]}...")
    print(f"  Word count: {len(row.response.split())}")
    print(f"  Conciseness Score: {row.response_conciseness:.3f}")

### Option B: Custom Metric with LLM Judge

For more nuanced evaluation, you can create a metric that uses an LLM to judge quality based on custom criteria.

In [ ]:
from ragas.metrics.base import SingleTurnMetric, MetricType
from dataclasses import dataclass, field
import typing as t


@dataclass
class ToneAppropriatenessMetric(SingleTurnMetric):
    """Custom LLM-as-judge metric that evaluates whether the response
    has an appropriate professional tone for customer support use.
    """
    name: str = "tone_appropriateness"
    _required_columns: t.Dict[MetricType, t.Set[str]] = field(
        default_factory=lambda: {MetricType.SINGLE_TURN: {"response", "user_input"}}
    )
    llm: t.Any = None
    
    async def _single_turn_ascore(self, sample: SingleTurnSample, callbacks=None) -> float:
        """Use LLM to judge tone appropriateness."""
        prompt = f"""Evaluate whether the following response has an appropriate professional 
tone for a customer support assistant. Consider:
- Is the language professional and clear?
- Does it avoid slang, excessive informality, or inappropriate humor?
- Is it helpful without being condescending?
- Does it maintain objectivity?

Question: {sample.user_input}
Response: {sample.response}

Rate the tone on a scale from 0 to 1 where:
- 1.0 = Perfectly professional and appropriate
- 0.5 = Acceptable but could be improved
- 0.0 = Inappropriate tone

Return ONLY a number between 0 and 1, nothing else."""

        from langchain_core.messages import HumanMessage
        result = await self.llm.agenerate([[HumanMessage(content=prompt)]])
        score_text = result.generations[0][0].text.strip()
        
        try:
            score = float(score_text)
            return max(0.0, min(1.0, score))
        except ValueError:
            return 0.5  # Default if parsing fails


# Test the LLM-judge metric
tone_metric = ToneAppropriatenessMetric(
    llm=ChatOpenAI(model="gpt-4o-mini", temperature=0)
)

tone_test_samples = [
    SingleTurnSample(
        user_input="What is the return policy?",
        response="Acme Corp offers a 30-day return policy on all products. Items must be unused and in original packaging with the receipt. Refunds are processed within 5-7 business days.",
        retrieved_contexts=["Return policy info."],
    ),
    SingleTurnSample(
        user_input="What is the return policy?",
        response="lol its like 30 days or somethin, just send it back i guess. idk just check the website bro",
        retrieved_contexts=["Return policy info."],
    ),
]

tone_dataset = EvaluationDataset(samples=tone_test_samples)
tone_results = evaluate(dataset=tone_dataset, metrics=[tone_metric])

tone_df = tone_results.to_pandas()
print("Tone Appropriateness Scores:")
print("=" * 60)
for idx, row in tone_df.iterrows():
    print(f"\n  Response: {row['response'][:80]}...")
    print(f"  Tone Score: {row['tone_appropriateness']:.3f}")

-
## Part 5: RAGAS with LangChain Integration

RAGAS integrates naturally with LangChain. Here we show how to wrap a LangChain chain and evaluate it with RAGAS.

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Build a LangChain RAG chain
lc_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful customer support assistant for Acme Corp. Answer based only on the provided context. Be concise."),
    ("user", "Context:\n{context}\n\nQuestion: {question}"),
])

# Create the chain
chain = prompt_template | lc_llm | StrOutputParser()

# Wrap the retriever to use our Qdrant retriever
def langchain_retrieve_and_generate(query):
    """Retrieve contexts and generate answer using LangChain chain."""
    contexts = retrieve(query, "acme_default", top_k=3)
    context_str = "\n\n".join(contexts)
    answer = chain.invoke({"context": context_str, "question": query})
    return {"answer": answer, "contexts": contexts}

# Test the LangChain chain
test = langchain_retrieve_and_generate("What is Acme Corp's return policy?")
print(f"Answer: {test['answer']}")
print(f"Contexts: {len(test['contexts'])} retrieved")

In [ ]:
# Evaluate the LangChain chain with RAGAS
lc_test_queries = [
    {"query": "What is the return window for electronics?", "reference": "Electronics have a 15-day return window instead of the standard 30 days."},
    {"query": "How much does expedited shipping cost?", "reference": "Expedited Shipping takes 2-3 business days and costs $12.99."},
    {"query": "What are the benefits of the Gold loyalty tier?", "reference": "Gold tier (1000+ points/year) adds priority customer support and exclusive product previews."},
    {"query": "What warranty comes with Acme products?", "reference": "All Acme Corp branded products come with a 1-year limited warranty."},
]

lc_samples = []
for tc in lc_test_queries:
    result = langchain_retrieve_and_generate(tc["query"])
    sample = SingleTurnSample(
        user_input=tc["query"],
        response=result["answer"],
        reference=tc["reference"],
        retrieved_contexts=result["contexts"],
    )
    lc_samples.append(sample)

lc_dataset = EvaluationDataset(samples=lc_samples)

lc_results = evaluate(
    dataset=lc_dataset,
    metrics=[
        Faithfulness(llm=evaluator_llm),
        ResponseRelevancy(llm=evaluator_llm, embeddings=evaluator_embeddings),
        FactualCorrectness(llm=evaluator_llm),
    ],
)

lc_df = lc_results.to_pandas()
print("LangChain Chain Evaluation Results:")
print("=" * 60)
print(f"  Faithfulness (mean):        {lc_df['faithfulness'].mean():.3f}")
print(f"  Answer Relevancy (mean):    {lc_df['answer_relevancy'].mean():.3f}")
print(f"  Factual Correctness (mean): {lc_df['factual_correctness'].mean():.3f}")
lc_df

-
## Part 6: Comparing Multiple RAG Configurations

One of the most valuable uses of evaluation metrics is **comparing different RAG configurations**. Let us test how different settings affect quality:

1. **Chunk size**: 200 vs 500 vs 1000 characters
2. **Top-K**: 2 vs 5 vs 8 retrieved chunks

In [ ]:
# Define the test queries for comparison
comparison_queries = [
    {"query": "What is Acme Corp's return policy for regular items?", "reference": "Acme Corp offers a 30-day return policy. Items must be unused, in original packaging, with receipt."},
    {"query": "What does the warranty cover?", "reference": "1-year limited warranty covering defects in materials and workmanship. Does not cover accidents or misuse."},
    {"query": "How much does overnight shipping cost?", "reference": "Overnight Shipping is available for $24.99 and delivers the next business day if ordered before 2 PM EST."},
    {"query": "How can I contact customer support?", "reference": "Phone (1-800-ACME-HELP, M-F 8AM-8PM EST), email (support@acmecorp.com), and live chat (M-Sat 9AM-6PM EST)."},
    {"query": "How does the loyalty rewards program work?", "reference": "Free to join, 1 point per dollar. 100 points = $5 off. Silver tier at 500+ points, Gold at 1000+ points."},
]

print(f"Using {len(comparison_queries)} queries for configuration comparison")

In [ ]:
# Test different chunk sizes
chunk_sizes = [200, 500, 1000]
chunk_size_results = {}

for chunk_size in chunk_sizes:
    collection_name = f"acme_chunk_{chunk_size}"
    n_chunks = build_qdrant_collection(ACME_DOCUMENTS, collection_name, chunk_size=chunk_size)
    print(f"\nChunk size {chunk_size}: {n_chunks} chunks created")
    
    samples = []
    for tc in comparison_queries:
        result = rag_pipeline(tc["query"], collection_name=collection_name, top_k=3)
        sample = SingleTurnSample(
            user_input=tc["query"],
            response=result["answer"],
            reference=tc["reference"],
            retrieved_contexts=result["contexts"],
        )
        samples.append(sample)
    
    dataset = EvaluationDataset(samples=samples)
    results = evaluate(
        dataset=dataset,
        metrics=[
            Faithfulness(llm=evaluator_llm),
            ResponseRelevancy(llm=evaluator_llm, embeddings=evaluator_embeddings),
            LLMContextRecall(llm=evaluator_llm),
        ],
    )
    
    results_df = results.to_pandas()
    chunk_size_results[chunk_size] = {
        "faithfulness": results_df["faithfulness"].mean(),
        "answer_relevancy": results_df["answer_relevancy"].mean(),
        "context_recall": results_df["llm_context_recall"].mean(),
    }
    print(f"  Faithfulness: {chunk_size_results[chunk_size]['faithfulness']:.3f}")
    print(f"  Relevancy:    {chunk_size_results[chunk_size]['answer_relevancy']:.3f}")
    print(f"  Ctx Recall:   {chunk_size_results[chunk_size]['context_recall']:.3f}")

In [ ]:
# Test different top-K values
top_k_values = [2, 5, 8]
top_k_results = {}

for top_k in top_k_values:
    print(f"\nTesting top_k={top_k}")
    
    samples = []
    for tc in comparison_queries:
        result = rag_pipeline(tc["query"], collection_name="acme_default", top_k=top_k)
        sample = SingleTurnSample(
            user_input=tc["query"],
            response=result["answer"],
            reference=tc["reference"],
            retrieved_contexts=result["contexts"],
        )
        samples.append(sample)
    
    dataset = EvaluationDataset(samples=samples)
    results = evaluate(
        dataset=dataset,
        metrics=[
            Faithfulness(llm=evaluator_llm),
            ResponseRelevancy(llm=evaluator_llm, embeddings=evaluator_embeddings),
            LLMContextRecall(llm=evaluator_llm),
        ],
    )
    
    results_df = results.to_pandas()
    top_k_results[top_k] = {
        "faithfulness": results_df["faithfulness"].mean(),
        "answer_relevancy": results_df["answer_relevancy"].mean(),
        "context_recall": results_df["llm_context_recall"].mean(),
    }
    print(f"  Faithfulness: {top_k_results[top_k]['faithfulness']:.3f}")
    print(f"  Relevancy:    {top_k_results[top_k]['answer_relevancy']:.3f}")
    print(f"  Ctx Recall:   {top_k_results[top_k]['context_recall']:.3f}")

In [ ]:
# Summarize configuration comparison results
print("Configuration Comparison Summary")
print("=" * 70)

print("\n--- Chunk Size Comparison ---")
chunk_comparison_df = pd.DataFrame(chunk_size_results).T
chunk_comparison_df.index.name = "chunk_size"
print(chunk_comparison_df.round(3))

best_chunk = chunk_comparison_df.mean(axis=1).idxmax()
print(f"\nBest chunk size: {best_chunk} (highest average across all metrics)")

print("\n--- Top-K Comparison ---")
topk_comparison_df = pd.DataFrame(top_k_results).T
topk_comparison_df.index.name = "top_k"
print(topk_comparison_df.round(3))

best_topk = topk_comparison_df.mean(axis=1).idxmax()
print(f"\nBest top_k: {best_topk} (highest average across all metrics)")

-
## Part 7: RAGAS vs DeepEval Side-by-Side

Let us run equivalent metrics from both RAGAS and DeepEval on the same test data and compare the scores. This helps you understand:
- How the frameworks measure similar concepts differently
- Whether scores are comparable across frameworks
- Which framework to prefer for specific use cases

In [ ]:
# Shared test data for side-by-side comparison
comparison_data = [
    {
        "query": "What is Acme Corp's return policy?",
        "reference": "Acme Corp offers a 30-day return policy. Items must be unused, in original packaging, with receipt. Refunds processed in 5-7 business days.",
    },
    {
        "query": "How much does overnight shipping cost?",
        "reference": "Overnight Shipping is available for $24.99 and delivers the next business day if ordered before 2 PM EST.",
    },
    {
        "query": "What payment methods are accepted?",
        "reference": "Acme Corp accepts Visa, MasterCard, American Express, Discover, PayPal, Apple Pay, Google Pay, and Acme Gift Cards.",
    },
    {
        "query": "What warranty do Acme products come with?",
        "reference": "All Acme Corp branded products come with a 1-year limited warranty covering defects in materials and workmanship.",
    },
]

# Run through our RAG pipeline
shared_results = []
for tc in comparison_data:
    result = rag_pipeline(tc["query"])
    shared_results.append({
        "query": tc["query"],
        "response": result["answer"],
        "reference": tc["reference"],
        "contexts": result["contexts"],
    })

print(f"Prepared {len(shared_results)} shared test cases")

In [ ]:
# RAGAS evaluation
ragas_samples = [
    SingleTurnSample(
        user_input=sr["query"],
        response=sr["response"],
        reference=sr["reference"],
        retrieved_contexts=sr["contexts"],
    )
    for sr in shared_results
]

ragas_dataset = EvaluationDataset(samples=ragas_samples)

ragas_eval = evaluate(
    dataset=ragas_dataset,
    metrics=[
        Faithfulness(llm=evaluator_llm),
        ResponseRelevancy(llm=evaluator_llm, embeddings=evaluator_embeddings),
        FactualCorrectness(llm=evaluator_llm),
    ],
)

ragas_df = ragas_eval.to_pandas()
print("RAGAS Results:")
print(ragas_df[["user_input", "faithfulness", "answer_relevancy", "factual_correctness"]].round(3))

In [ ]:
# DeepEval evaluation on the same data
from deepeval.test_case import LLMTestCase
from deepeval.metrics import (
    FaithfulnessMetric,
    AnswerRelevancyMetric,
    GEval,
)
from deepeval import evaluate as deepeval_evaluate

# Create DeepEval test cases
de_test_cases = []
for sr in shared_results:
    tc = LLMTestCase(
        input=sr["query"],
        actual_output=sr["response"],
        expected_output=sr["reference"],
        retrieval_context=sr["contexts"],
    )
    de_test_cases.append(tc)

# Define DeepEval metrics
de_faithfulness = FaithfulnessMetric(model="gpt-4o-mini", threshold=0.7)
de_relevancy = AnswerRelevancyMetric(model="gpt-4o-mini", threshold=0.7)
de_correctness = GEval(
    name="Correctness",
    model="gpt-4o-mini",
    evaluation_params=[
        {"name": "actual_output", "type": "str"},
        {"name": "expected_output", "type": "str"},
    ],
    evaluation_steps=[
        "Compare the actual output with the expected output.",
        "Check if the key facts in the expected output are present in the actual output.",
        "Score 1.0 if all facts match, lower for missing or incorrect facts.",
    ],
    threshold=0.7,
)

# Run DeepEval metrics on each test case
de_scores = {"faithfulness": [], "relevancy": [], "correctness": []}

for tc in de_test_cases:
    de_faithfulness.measure(tc)
    de_relevancy.measure(tc)
    de_correctness.measure(tc)
    
    de_scores["faithfulness"].append(de_faithfulness.score)
    de_scores["relevancy"].append(de_relevancy.score)
    de_scores["correctness"].append(de_correctness.score)

de_comparison_df = pd.DataFrame({
    "Query": [sr["query"] for sr in shared_results],
    "DE_Faithfulness": de_scores["faithfulness"],
    "DE_Relevancy": de_scores["relevancy"],
    "DE_Correctness": de_scores["correctness"],
})

print("DeepEval Results:")
print(de_comparison_df.round(3))

In [ ]:
# Side-by-side comparison
comparison_df = pd.DataFrame({
    "Query": [sr["query"][:45] for sr in shared_results],
    "RAGAS_Faith": ragas_df["faithfulness"].values,
    "DE_Faith": de_scores["faithfulness"],
    "RAGAS_Relev": ragas_df["answer_relevancy"].values,
    "DE_Relev": de_scores["relevancy"],
    "RAGAS_Correct": ragas_df["factual_correctness"].values,
    "DE_Correct": de_scores["correctness"],
})

print("\nSide-by-Side Comparison: RAGAS vs DeepEval")
print("=" * 90)
print(comparison_df.round(3).to_string(index=False))

print("\n--- Average Scores ---")
print(f"  Faithfulness:  RAGAS={ragas_df['faithfulness'].mean():.3f}  |  DeepEval={np.mean(de_scores['faithfulness']):.3f}")
print(f"  Relevancy:     RAGAS={ragas_df['answer_relevancy'].mean():.3f}  |  DeepEval={np.mean(de_scores['relevancy']):.3f}")
print(f"  Correctness:   RAGAS={ragas_df['factual_correctness'].mean():.3f}  |  DeepEval={np.mean(de_scores['correctness']):.3f}")

### Framework Comparison Discussion

| Aspect | RAGAS | DeepEval |
|--------|-------|----------|
| **Faithfulness** | Claim extraction + NLI (research-backed) | Similar approach but with different prompt engineering |
| **Answer Relevancy** | Reverse question generation + embedding similarity | LLM-as-judge approach with structured evaluation |
| **Correctness** | Factual F1 with claim-level comparison | G-Eval with custom criteria (more flexible) |
| **Testset Generation** | Built-in with evolution types | Synthesizer with different generation strategies |
| **Custom Metrics** | SingleTurnMetric subclass | BaseMetric subclass |
| **Integration** | Strong LangChain integration | Pytest-native, CI/CD friendly |

**When to use RAGAS**: Research-oriented evaluation, LangChain-based pipelines, when you need testset generation.

**When to use DeepEval**: CI/CD integration, pytest-based testing, when you need 50+ metric types including agentic evaluation.

**Best practice**: Use both frameworks -- they complement each other and provide different perspectives on quality.

## Summary

In this notebook we covered advanced RAGAS features:

1. **Testset Generation** -- Automatically generate evaluation data from documents with different evolution types
2. **Noise Sensitivity** -- Measure robustness to irrelevant context
3. **Semantic Similarity** -- Embedding-based answer comparison
4. **Custom Metrics** -- Build programmatic and LLM-judge metrics
5. **LangChain Integration** -- Evaluate LangChain chains directly
6. **Configuration Comparison** -- Find optimal chunk size and top-K
7. **RAGAS vs DeepEval** -- Side-by-side framework comparison

**Next notebook**: Deep dive into faithfulness, hallucination, and grounding.